# 01 · Download Sentinel-2 statistics per topographic zone

Searches Sentinel-2 L2A scenes on Microsoft Planetary Computer (no account needed). For every date and topographic zone of the AOI it stores the NDVI, GRVI, PSRI and RCC statistics of the clear pixels, plus cloud and quality variables.

- **Settings:** `config.py` (AOI, `FIRST_YEAR`–`LAST_YEAR`, masking and zone settings, `SITE_AREAS`, `SITE_RADIUS_M`)
- **Reads:** `LOCAL_PHENO_FILE` (site coordinates, section 6 only)
- **Writes** to `cache/<AOI>_zones/`:
  - `stats_<year>.csv`: one row per date and zone
  - `zones.csv` and `zones.tif`: zone attributes and zone raster
  - `settings.json`: the settings the cache was built with
- **Writes** to `cache/<AOI>_sites/` (section 6): the same statistics for the AOI pixels around the local sites of `SITE_AREAS`
- **Next:** `04_phenology_metrics` (and `03_download_climate`, which needs `zones.tif`)

Years that already have a CSV are skipped, so raising `LAST_YEAR` only downloads the new years. A year still in progress is cached as far as it goes; delete its CSV to fetch newer scenes.

In [ ]:
import json
import warnings

import dask
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import planetary_computer
import pystac_client
import rasterio
import rioxarray
import stackstac
import xarray as xr
from dask.diagnostics import ProgressBar
from rasterio.enums import Resampling
from rasterio.features import geometry_mask, shapes
from rasterio.warp import transform_bounds
from rioxarray.exceptions import NoDataInBounds, OneDimensionalRaster
from rioxarray.merge import merge_arrays
from shapely.geometry import shape

import importlib
import config, phenology_utils
importlib.reload(config); importlib.reload(phenology_utils)  # pick up edits to config.py without a kernel restart
from config import *
from phenology_utils import cached_years, incomplete_years, index_frame, load_stats, load_zone_table

DOWNLOAD = True     # False: only check the cache
DASK_WORKERS = 8    # parallel reads; lower it if memory runs short
REFRESH_MISSING_INDICES = True   # re-download cached years whose CSV lacks an index of INDICES
print(f"AOI: {AOI_NAME} | years {FIRST_YEAR}-{LAST_YEAR} | cache: {CACHE_DIR}")

## 1. AOI

In [ ]:
def aoi_from_geotiff(path, values=None, band=1):
    # Vectorise the selected mask pixels so the rest of the notebook can treat them like a polygon
    with rasterio.open(path) as src:
        data = src.read(band, masked=True)
        transform, crs = src.transform, src.crs
    if crs is None:
        raise ValueError(f"{path} has no CRS; assign one (e.g. with gdal_edit.py -a_srs) first")
    valid = ~np.ma.getmaskarray(data) & np.isfinite(data.data)
    keep = valid & (np.isin(data.data, values) if values is not None else data.data != 0)
    if not keep.any():
        raise ValueError(f"No pixels in {path} match AOI_RASTER_VALUES={values}")
    geoms = [shape(g) for g, _ in shapes(keep.astype("uint8"), mask=keep, transform=transform)]
    print(f"{int(keep.sum())} raster pixels -> {len(geoms)} polygon part(s)")
    return gpd.GeoDataFrame(geometry=geoms, crs=crs)


if not AOI_PATH.exists():
    raise FileNotFoundError(f"{AOI_PATH} not found")
if AOI_PATH.suffix.lower() in {".tif", ".tiff"}:
    aoi = aoi_from_geotiff(AOI_PATH, AOI_RASTER_VALUES, AOI_RASTER_BAND)
else:
    aoi = gpd.read_file(AOI_PATH)
aoi = aoi.dissolve()
aoi_wgs84 = aoi.to_crs(4326)
aoi.explore(color="darkgreen", fill=False, linewidth=2, tiles="OpenStreetMap", tooltip=False)

## 2. Scene search and lazy data cube

Planetary Computer signed URLs expire after about an hour, so every year gets its own search right before its download. All scenes are returned regardless of cloud cover; the filters are applied when the cache is loaded (`04`). Nothing is downloaded until the statistics are computed.

In [ ]:
def search_scenes(start, end, area=None):
    # Scenes over area (WGS84 geometry; default: the AOI). Fresh token: the cached one is only renewed in its last
    # minute and can expire mid-download
    planetary_computer.sas.TOKEN_CACHE.clear()
    catalog = pystac_client.Client.open("https://planetarycomputer.microsoft.com/api/stac/v1",
                                        modifier=planetary_computer.sign_inplace)
    # Convex hull keeps the request small; a GeoTIFF mask dissolves into thousands of pixel outlines
    area = aoi_wgs84.geometry.iloc[0].convex_hull if area is None else area
    search = catalog.search(collections=["sentinel-2-l2a"], intersects=area, datetime=f"{start}/{end}")
    return search.item_collection()


def item_epsg(item):
    p = item.properties
    if p.get("proj:epsg"):
        return int(p["proj:epsg"])
    return int(str(p["proj:code"]).split(":")[-1])


def build_cube(items, bounds_latlon=None):
    # Most common UTM zone of the scenes; plain int, otherwise stackstac's out_epsg assertion fails
    epsg = int(pd.Series([item_epsg(i) for i in items]).mode()[0])
    aoi_utm = aoi.to_crs(epsg)
    if INNER_BUFFER:
        aoi_utm = aoi_utm.buffer(-INNER_BUFFER).to_frame("geometry")
    ds = stackstac.stack(
        items,
        assets=["B02", "B03", "B04", "B06", "B08", "SCL"],   # blue, green, red, red edge 2 (20 m), NIR, SCL
        resolution=RESOLUTION,
        epsg=epsg,
        bounds_latlon=tuple(aoi_wgs84.total_bounds) if bounds_latlon is None else bounds_latlon,
        resampling=0,          # nearest neighbour, required for the SCL class band
        chunksize=1024,
        xy_coords="center",    # pixel centres, as rioxarray assumes (default top-left shifts the mask by 5 m)
    )
    return ds.rio.write_crs(epsg), aoi_utm

## 3. Topographic zones

The AOI pixels are split into **`ELEV_STEP` elevation bands × north / south facing**, plus "flat" for slopes under `FLAT_SLOPE`.

- Elevation comes from the Copernicus DEM GLO-30. Slope and aspect are computed at 30 m and resampled bilinearly to the 10 m grid.
- Aspect is carried as northness = cos(aspect) and eastness = sin(aspect), so that 359° and 1° do not average to 180°. A pixel is north facing if northness ≥ 0.
- Zones with fewer than `MIN_ZONE_PIXELS` pixels are dropped. The zone `"all"` is always the whole AOI.

In [ ]:
ASPECT_CODES = {"N": 0, "S": 1, "flat": 2}   # aspect classes as stored in zones.tif
ZONE_NODATA = -9999


def dem_30m(epsg, bounds):
    # Copernicus DEM tiles covering bounds (x0, y0, x1, y1 in epsg), mosaicked and reprojected to epsg at 30 m
    catalog = pystac_client.Client.open("https://planetarycomputer.microsoft.com/api/stac/v1",
                                        modifier=planetary_computer.sign_inplace)
    lonlat = transform_bounds(f"EPSG:{epsg}", "EPSG:4326", *bounds)
    tiles = []
    for item in catalog.search(collections=[DEM_COLLECTION], bbox=lonlat).item_collection():
        tile = rioxarray.open_rasterio(item.assets["data"].href, masked=True).squeeze("band", drop=True)
        try:
            tiles.append(tile.rio.clip_box(*lonlat, crs="EPSG:4326"))
        except (NoDataInBounds, OneDimensionalRaster):   # tile only touches the box
            pass
    if not tiles:
        raise ValueError(f"No {DEM_COLLECTION} tiles over the AOI")
    dem = merge_arrays(tiles) if len(tiles) > 1 else tiles[0]
    return dem.rio.reproject(f"EPSG:{epsg}", resolution=30, resampling=Resampling.bilinear)


def zone_layers(ds, inside):
    # Zone masks {"all": inside, "1800_N": ...}, zone table and zone raster (for zones.tif) on the cube grid
    epsg = ds.rio.crs.to_epsg()
    x0, y0, x1, y1 = ds.rio.bounds()
    dem = dem_30m(epsg, (x0 - 100, y0 - 100, x1 + 100, y1 + 100))   # padded, so slope is defined at the edges

    # Passing the coordinates gives gradients in m/m and handles the descending y axis
    z = dem.values.astype("float64")
    dz_dy, dz_dx = np.gradient(z, dem.y.values, dem.x.values)
    grad = np.hypot(dz_dx, dz_dy)
    # Aspect = compass direction of steepest descent (0 = N, 90 = E)
    with np.errstate(invalid="ignore", divide="ignore"):
        northness, eastness = -dz_dy / grad, -dz_dx / grad
    flat = grad == 0
    northness[flat] = eastness[flat] = 0
    layers = xr.Dataset({
        name: dem.copy(data=v.astype("float32")).rio.write_nodata(np.nan, encoded=False)
        for name, v in {"elevation": z, "slope": np.degrees(np.arctan(grad)),
                        "northness": northness, "eastness": eastness}.items()})

    grid = layers.rio.reproject_match(ds.isel(time=0, band=0, drop=True), resampling=Resampling.bilinear)
    grid = grid.assign_coords(x=ds.x, y=ds.y)

    has_dem = inside & grid.elevation.notnull() & grid.slope.notnull()
    elev_band = np.floor(grid.elevation / ELEV_STEP) * ELEV_STEP
    aspect = xr.where(grid.slope < FLAT_SLOPE, ASPECT_CODES["flat"],
                      xr.where(grid.northness >= 0, ASPECT_CODES["N"], ASPECT_CODES["S"]))

    def summary(zone, mask, band, asp):
        means = {f"{short}_mean": float(grid[var].where(mask).mean())
                 for short, var in [("elev", "elevation"), ("slope", "slope"),
                                    ("northness", "northness"), ("eastness", "eastness")]}
        return {"zone": zone, "elev_band": band, "aspect": asp, "n_pixels": int(mask.sum()), **means}

    zones, rows = {"all": inside}, [summary("all", inside, np.nan, "all")]
    zone_id = xr.full_like(elev_band, ZONE_NODATA)   # row number in the zone table; nodata = only in "all"
    for band in np.unique(elev_band.values[has_dem.values]):
        for asp, code in ASPECT_CODES.items():
            mask = has_dem & (elev_band == band) & (aspect == code)
            if int(mask.sum()) >= MIN_ZONE_PIXELS:
                zone = f"{int(band)}_{asp}"
                zone_id = zone_id.where(~mask, len(rows))
                zones[zone] = mask
                rows.append(summary(zone, mask, int(band), asp))
    zone_table = pd.DataFrame(rows).set_index("zone")
    n_inside, n_kept = int(inside.sum()), int(zone_table.n_pixels.iloc[1:].sum())
    print(f"  {len(zones) - 1} zones with >= {MIN_ZONE_PIXELS} pixels; {n_inside - n_kept} of {n_inside} "
          "AOI pixels are only in 'all' (small zone or no DEM)")

    # zones.tif bands: elevation (whole extent), elevation band, aspect class, zone row (AOI pixels only)
    zone_raster = (xr.concat([grid.elevation.round(), elev_band.where(has_dem), aspect.where(has_dem), zone_id],
                             dim="band")
                   .fillna(ZONE_NODATA).astype("int16").assign_coords(band=[1, 2, 3, 4])
                   .rio.write_nodata(ZONE_NODATA).rio.write_crs(epsg))
    zone_raster.attrs["long_name"] = ("elevation", "elev_band", "aspect", "zone_id")
    return zones, zone_table, zone_raster

## 4. Indices on clear pixels

- **Baseline offset:** scenes with processing baseline 04.00 or later (from 25 Jan 2022, plus a few reprocessed older scenes) carry a +1000 reflectance offset. The Planetary Computer collection is not harmonised, so the offset is subtracted per scene according to its `s2:processing_baseline`; otherwise the series would jump.
- **Indices** (`INDICES` in `config.py`), computed on pixels with SCL in `SCL_KEEP` inside the AOI:
  - NDVI = (B08 − B04) / (B08 + B04)
  - GRVI = (B03 − B04) / (B03 + B04), green–red balance; falls as needles turn yellow (Motohka et al. 2010)
  - PSRI = (B04 − B02) / B06, rises with the carotenoid / chlorophyll ratio during senescence (Merzlyak et al. 1999)
  - RCC = B04 / (B02 + B03 + B04), the red chromatic coordinate used for autumn colour in PhenoCam studies
    (Sonnentag et al. 2012; Richardson et al. 2018); rises as the needles turn yellow
- Same-day scenes from overlapping tiles are merged.

In [ ]:
def harmonised_bands(ds):
    offset = xr.where(ds["s2:processing_baseline"].astype(float) >= 4, 1000, 0)
    bands = {name: (ds.sel(band=b) - offset).clip(min=0)
             for name, b in [("blue", "B02"), ("green", "B03"), ("red", "B04"), ("red_edge", "B06"), ("nir", "B08")]}
    bands["scl"] = ds.sel(band="SCL")
    return bands


def polygon_mask(ds, aoi_utm):
    inside = ~geometry_mask(aoi_utm.geometry, out_shape=(ds.sizes["y"], ds.sizes["x"]),
                            transform=ds.rio.transform(), all_touched=False)
    return xr.DataArray(inside, dims=("y", "x"), coords={"y": ds.y, "x": ds.x})


def index_cube(num, den, valid):
    # Lazy ratio on valid pixels (zero denominators -> NaN), same-day scenes averaged
    num, den = num.where(valid), den.where(valid)
    idx = (num / den).where(den != 0)
    idx = idx.groupby(idx.time.dt.floor("D")).mean(skipna=True).rename(floor="time")
    # One chunk per date over the whole grid, so the spatial median works on the lazy cube
    return idx.chunk({"time": 1, "y": -1, "x": -1})


# Numerator and denominator of each index from the harmonised bands
INDEX_FORMULAS = {
    "ndvi": lambda b: (b["nir"] - b["red"], b["nir"] + b["red"]),
    "grvi": lambda b: (b["green"] - b["red"], b["green"] + b["red"]),
    "psri": lambda b: (b["red"] - b["blue"], b["red_edge"]),
    "rcc": lambda b: (b["red"], b["blue"] + b["green"] + b["red"]),
}


def index_cubes(ds, mask):
    b = harmonised_bands(ds)
    valid = b["scl"].isin(SCL_KEEP) & mask
    return {name.lower(): index_cube(*INDEX_FORMULAS[name.lower()](b), valid) for name in INDICES}


def scl_fractions(scl, zones, n_pixels):
    # Share of each zone's pixels per SCL group and date. Same-day scenes: a pixel counts for a class if any
    # scene shows it, but as nodata only if no scene has data there.
    day = scl.time.dt.floor("D")
    out = {}
    for group, classes in SCL_GROUPS.items():
        if group == "nodata":
            hit = (scl.isin(classes) | scl.isnull()).astype("uint8").groupby(day).min()
        else:
            hit = scl.isin(classes).astype("uint8").groupby(day).max()
        hit = hit.rename(floor="time")
        for zone, mask in zones.items():
            out[(zone, f"scl_{group}_fraction")] = hit.where(mask, 0).sum(["x", "y"]) / n_pixels[zone]
    return out

## 5. Download per year

Every year in `FIRST_YEAR`–`LAST_YEAR` (up to the current year) gets its own search, download and `stats_<year>.csv`. A cached year whose CSV lacks one of `INDICES` is downloaded again if `REFRESH_MISSING_INDICES` is set. All zones are computed in the same pass, so each band is downloaded only once.

A year downloaded before its autumn is over (cache ends before `SEASON_COMPLETE_DOY`) is reported below; `04` leaves it out until its CSV is deleted and downloaded again.

Each row holds, for one date and zone (`"all"` = whole AOI):
- `<index>_mean/median/std`
- `<index>_valid_fraction`: share of the zone's pixels that are clear and valid
- `scene_cloud_cover`: tile cloud cover in %, max over same-day tiles
- `scl_<group>_fraction`: share of the zone's pixels per SCL class group

The first downloaded year also writes `zones.csv` and `zones.tif`. The cache is only valid for the settings stored in `settings.json`; this cell refuses to mix caches. After an error, simply re-run the cell.

In [ ]:
cache_settings = {
    # relative to the project folder, so the cache stays valid in another clone of the repository
    "aoi_path": AOI_PATH.relative_to(ROOT).as_posix() if AOI_PATH.is_relative_to(ROOT) else str(AOI_PATH),
    "aoi_raster_values": AOI_RASTER_VALUES, "aoi_raster_band": AOI_RASTER_BAND,
    "scl_keep": SCL_KEEP, "scl_groups": SCL_GROUPS, "resolution": RESOLUTION, "inner_buffer": INNER_BUFFER,
    "all_scenes": True,
    "dem": DEM_COLLECTION, "elev_step": ELEV_STEP, "flat_slope": FLAT_SLOPE, "min_zone_pixels": MIN_ZONE_PIXELS,
}


def check_settings(cache_dir, settings):
    # A cache is only valid for the settings it was built with; a new cache records them in settings.json
    settings_file = cache_dir / "settings.json"
    if settings_file.exists():
        cached_settings = json.loads(settings_file.read_text())
        if cached_settings != settings:
            raise ValueError(f"{cache_dir} was built with different settings:\n{cached_settings}\n"
                             "Change the settings back or delete that folder to rebuild it.")
    elif DOWNLOAD:
        cache_dir.mkdir(parents=True, exist_ok=True)
        settings_file.write_text(json.dumps(settings, indent=1))
    else:
        raise FileNotFoundError(f"No cache in {cache_dir}; run once with DOWNLOAD = True")


def date_stats(cube, n_inside):
    return {
        "mean": cube.mean(["x", "y"]),
        "median": cube.median(["x", "y"]),
        "std": cube.std(["x", "y"]),
        "valid_fraction": cube.notnull().sum(["x", "y"]) / n_inside,
    }


def save_zone_info(zone_table, zone_raster, year):
    # zones.csv / zones.tif come from the first downloaded year; later years should give the same zones
    table_file = CACHE_DIR / "zones.csv"
    if not table_file.exists():
        zone_table.to_csv(table_file)
        zone_raster.rio.to_raster(CACHE_DIR / "zones.tif", compress="deflate")
    elif not pd.read_csv(table_file, index_col="zone").n_pixels.equals(zone_table.n_pixels):
        warnings.warn(f"{year}: zone pixel counts differ from {table_file} (different UTM zone/grid?)")


def zone_frame(ds, items, inside, zones, n_pixels):
    # Statistics per date and zone (zones: {name: mask}, all within the mask inside)
    cubes = index_cubes(ds, inside)
    lazy = {}
    for zone, mask in zones.items():
        for name, cube in cubes.items():
            for stat, v in date_stats(cube.where(mask), n_pixels[zone]).items():
                lazy[(zone, f"{name}_{stat}")] = v
    lazy.update(scl_fractions(ds.sel(band="SCL"), zones, n_pixels))
    # One compute for all indices and zones: each band of each scene is downloaded once
    with ProgressBar(), dask.config.set(num_workers=DASK_WORKERS), warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)   # fully cloudy dates: NaN statistics
        (stats,) = dask.compute(lazy)
    dates = pd.to_datetime(cubes["ndvi"].time.values)
    item_days = pd.to_datetime([i.datetime for i in items], utc=True).tz_convert(None).floor("D")
    cover = pd.Series([i.properties.get("eo:cloud_cover", np.nan) for i in items], index=item_days)
    cover = cover.groupby(level=0).max().reindex(dates).values
    parts = []
    for zone in zones:
        part = pd.DataFrame({col: v.values for (z, col), v in stats.items() if z == zone}, index=dates)
        part.insert(0, "scene_cloud_cover", cover)
        part.insert(0, "zone", zone)
        parts.append(part)
    frame = pd.concat(parts).sort_index(kind="stable")
    frame.index.name = "date"
    return frame


def no_scenes():
    print("  no scenes")
    return pd.DataFrame(columns=["zone"], index=pd.DatetimeIndex([], name="date"))


def year_stats(year):
    items = search_scenes(f"{year}-01-01", f"{year}-12-31")
    if len(items) == 0:
        return no_scenes()
    ds, aoi_utm = build_cube(items)
    inside = polygon_mask(ds, aoi_utm)
    print(f"  {len(items)} scenes, {int(inside.sum())} pixels inside the AOI")
    zones, zone_table, zone_raster = zone_layers(ds, inside)
    save_zone_info(zone_table, zone_raster, year)
    return zone_frame(ds, items, inside, zones, zone_table.n_pixels)


def cache_complete(path):
    # The year's CSV exists and, unless REFRESH_MISSING_INDICES is off, has every index of INDICES
    if not path.exists():
        return False
    columns = pd.read_csv(path, nrows=0).columns
    return not REFRESH_MISSING_INDICES or all(f"{name.lower()}_median" in columns for name in INDICES)


def download_years(cache_dir, year_fn):
    # stats_<year>.csv of every year not cached yet; year_fn(year) -> the year's statistics
    failed = []
    for year in cached_years():
        path = cache_dir / f"stats_{year}.csv"
        if cache_complete(path):
            continue
        print(f"{year}:")
        try:
            year_fn(year).to_csv(path)
        except Exception as e:   # e.g. a server hiccup; the year is retried on the next run
            failed.append(year)
            print(f"  failed: {e!r}")
    if failed:
        print(f"Failed years {failed}: re-run this cell to retry them")


check_settings(CACHE_DIR, cache_settings)
if DOWNLOAD:
    download_years(CACHE_DIR, year_stats)

stats_all = load_stats()
print(f"Cached: {stats_all.index.year.min()}-{stats_all.index.year.max()}, "
      f"{stats_all.index.nunique()} dates, {stats_all.zone.nunique()} zones")
for year in incomplete_years(stats_all):
    print(f"{year}: cache ends {stats_all[stats_all.index.year == year].index.max():%d %b %Y}, autumn incomplete. "
          f"Delete {CACHE_DIR.name}/stats_{year}.csv after the season and re-run this cell.")

## 6. Areas around local sites

For a direct check of the dates against one site's observations (`04` section 7c), the AOI pixels within `SITE_RADIUS_M` of each site in `SITE_AREAS` get their own statistics. The site coordinates come from `LOCAL_PHENO_FILE` (LV95), the pixels and indices are those of sections 1–5, and the statistics have the same columns as `stats_<year>.csv` of the zones, with the site code as zone. Only a small box around the sites is read, so this takes minutes, not hours.

- **Writes** to `cache/<AOI>_sites/`: `stats_<year>.csv`, `zones.csv` (pixels and mean elevation of each site area, Copernicus DEM) and `settings.json`. After changing `SITE_AREAS` or `SITE_RADIUS_M`, delete that folder to rebuild it.

In [ ]:
site_settings = {**{k: cache_settings[k] for k in ("aoi_path", "aoi_raster_values", "aoi_raster_band", "scl_keep",
                                                  "scl_groups", "resolution", "inner_buffer", "all_scenes", "dem")},
                 "site_areas": SITE_AREAS, "site_radius_m": SITE_RADIUS_M}


def site_points():
    # Points of SITE_AREAS from LOCAL_PHENO_FILE (LV95)
    xy = pd.read_excel(LOCAL_PHENO_FILE).groupby("site_region")[["site_x_lv95", "site_y_lv95"]].first()
    missing = sorted(set(SITE_AREAS) - set(xy.index))
    if missing:
        raise ValueError(f"SITE_AREAS {missing} are not site codes of {LOCAL_PHENO_FILE.name}: {list(xy.index)}")
    xy = xy.loc[SITE_AREAS]
    return gpd.GeoSeries(gpd.points_from_xy(xy.site_x_lv95, xy.site_y_lv95), index=xy.index, crs=2056)


def save_site_info(ds, zones, n_pixels):
    # zones.csv of the site cache (from the first downloaded year): pixels and mean elevation per site area
    path = SITE_CACHE_DIR / "zones.csv"
    if path.exists():
        return
    x0, y0, x1, y1 = ds.rio.bounds()
    dem = dem_30m(ds.rio.crs.to_epsg(), (x0 - 100, y0 - 100, x1 + 100, y1 + 100))
    elev = dem.rio.reproject_match(ds.isel(time=0, band=0, drop=True), resampling=Resampling.bilinear)
    elev = elev.assign_coords(x=ds.x, y=ds.y)
    pd.DataFrame({"n_pixels": n_pixels, "elev_mean": {c: float(elev.where(m).mean()) for c, m in zones.items()},
                  "radius_m": SITE_RADIUS_M}).rename_axis("zone").to_csv(path)


def year_site_stats(year):
    circles = site_xy.buffer(SITE_RADIUS_M).to_crs(4326)
    items = search_scenes(f"{year}-01-01", f"{year}-12-31", circles.union_all().convex_hull)
    if len(items) == 0:
        return no_scenes()
    ds, aoi_utm = build_cube(items, tuple(circles.total_bounds))
    inside = polygon_mask(ds, aoi_utm)
    centres = site_xy.to_crs(ds.rio.crs)
    zones = {code: inside & (np.hypot(ds.x - p.x, ds.y - p.y) <= SITE_RADIUS_M) for code, p in centres.items()}
    n_pixels = pd.Series({code: int(m.sum()) for code, m in zones.items()})
    if (n_pixels == 0).any():
        raise ValueError(f"No AOI pixels within {SITE_RADIUS_M} m of {list(n_pixels.index[n_pixels == 0])}")
    print(f"  {len(items)} scenes, AOI pixels within {SITE_RADIUS_M} m: {n_pixels.to_dict()}")
    save_site_info(ds, zones, n_pixels)
    return zone_frame(ds, items, xr.concat(list(zones.values()), "site").any("site"), zones, n_pixels)


if SITE_AREAS:
    site_xy = site_points()
    check_settings(SITE_CACHE_DIR, site_settings)
    if DOWNLOAD:
        download_years(SITE_CACHE_DIR, year_site_stats)
    site_stats_all = load_stats(SITE_CACHE_DIR)
    display(load_zone_table(SITE_CACHE_DIR).round(1))
    print(f"Cached: {site_stats_all.index.year.min()}-{site_stats_all.index.year.max()}, "
          f"{site_stats_all.index.nunique()} dates, sites {sorted(site_stats_all.zone.unique())}")